# 05c — 第五讲基础习题：特征工程、SHAP 与三维分子巩固练习

本套习题帮助你一步步巩固第五讲（`05a`–`05b`）中特征选择（方差过滤 / 共线性剪枝 / RFE）、Shapley 值与 SHAP 归因、ETKDG 三维构象生成、3D 形状描述符以及 PubChem API 抓取的核心操作。每道题只需填写 **2–6 行代码**，并附有预期数值提示供你自查。完成后可对照 `solutions/05_exercises_solutions.ipynb` 查看完整解答。

---

## 0. 公共环境准备

In [ ]:
import os
import itertools
import math
import re
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors, Descriptors3D, rdMolDescriptors

RDLogger.DisableLog("rdApp.*")
plt.rcParams["figure.dpi"] = 110
np.random.seed(0)

def find_file(name):
    for path in (f"data/{name}", f"../data/{name}", name):
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"找不到 {name}")

<!-- ai4m-style -->
> **📘 课程统一绘图风格**：下面这格加载课程自带的 `ai4m.mplstyle`（配色与课程网站同源，无上右边框，仅横向浅网格，保存默认 300 dpi）。`C` 是课程配色，手动指定颜色时写 `color=C.crimson`。


In [ ]:
# ai4m-style：课程统一绘图风格（本地与 Colab 通用）
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))   # course/ 根目录
import ai4m_style as ai4m

C = ai4m.use()        # 应用 ai4m.mplstyle，返回课程配色


In [ ]:
# 本习题集统一使用的数据：AqSolDB 的 2,000 分子子集（17 个自带描述符）
df = pd.read_csv(find_file("solubility.csv"))
df = df[(df["MolWt"] > 20) & (df["MolWt"] < 800)].sample(2000, random_state=7).reset_index(drop=True)

FEATS = [c for c in df.columns if c not in ("Name", "SMILES", "Solubility")]
X = df[FEATS].copy()
y = df["Solubility"].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)

print(f"X: {X.shape[0]} 个分子 × {X.shape[1]} 个描述符")
print(FEATS)

---

## 第一部分：特征选择与 SHAP（对应 `05a`）

### 🏋️ 习题 1：找出“准常量”描述符

一个描述符如果在绝大多数分子上取值都相同，就几乎不携带区分信息。

请对训练集 `X_tr` 的每一列，计算 **最高频取值所占的比例**（提示：`s.value_counts(normalize=True).iloc[0]`），整理成一个按降序排列的 `pd.Series` 并打印前 5 项。

- **预期输出**：排在最前的是 `NumSaturatedRings`、`NumAliphaticRings` 这类计数型描述符，最高一项的占比在 `0.80` 以上（即 80% 以上的分子在这一列上取同一个值）。

In [ ]:
# 在下方填写你的代码：


### 🏋️ 习题 2：揪出高度共线的描述符对

1. 计算 `X_tr` 的**绝对值** Pearson 相关系数矩阵 `corr = X_tr.corr().abs()`；
2. 遍历矩阵的**上三角**（即 $i < j$，避免重复与对角线），把所有 $|r| > 0.95$ 的特征对连同相关系数收集到一个列表里；
3. 按相关系数从高到低打印。

- **预期输出**：`HeavyAtomCount` 与 `NumValenceElectrons`、`LabuteASA` 与 `MolWt` 等“分子尺寸”类描述符之间的 $|r|$ 都在 `0.95` 以上。

In [ ]:
# 在下方填写你的代码：


### 🏋️ 习题 3：用 RFE 把 17 个描述符压到 5 个

1. 用 `RFE(estimator=RandomForestRegressor(n_estimators=80, random_state=0, n_jobs=-1), n_features_to_select=5, step=2)` 在 `(X_tr, y_tr)` 上拟合；
2. 打印入选的 5 个描述符名（`X.columns[rfe.support_]`）；
3. 分别用**全部 17 个**特征和**入选的 5 个**特征训练 `RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1)`，打印两者的测试集 $R^2$。

- **思考**：特征数砍掉近三分之二，$R^2$ 变化了多少？（在这个数据集上你甚至可能看到 5 个特征**略胜于** 17 个——为什么剔除冗余特征反而可能提升泛化性能？）

In [ ]:
# 在下方填写你的代码：


### 🏋️ 习题 4：验证“选择性数据泄漏”

对比两种交叉验证写法（都用 `cv = KFold(5, shuffle=True, random_state=0)`，模型统一为 `RandomForestRegressor(n_estimators=120, random_state=0, n_jobs=-1)`）：

- **写法 A（有泄漏）**：先在**全量** `(X, y)` 上用 `RFE(..., n_features_to_select=4, step=3)` 选出 4 个特征，再只对这 4 列做 `cross_val_score`；
- **写法 B（正确）**：把同样的 `RFE` 和模型串进 `Pipeline`，对完整的 `(X, y)` 做 `cross_val_score`。

打印两者的平均 $R^2$ 及其差值。

- **说明**：本数据集样本多、信号强，两者差距会很小甚至为负。真正的危险发生在“样本少、候选特征上千”的场景——请在答案里用一句话解释为什么。

In [ ]:
# 在下方填写你的代码：


### 🏋️ 习题 5：手算两个特征的精确 Shapley 值

当只有 2 个特征 $F = \{A, B\}$ 时，Shapley 公式退化成一个可以手算的形式：

$$\phi_A = \tfrac{1}{2}\big[v(\{A\}) - v(\varnothing)\big] + \tfrac{1}{2}\big[v(\{A, B\}) - v(\{B\})\big]$$

（$\phi_B$ 完全对称）。下方代码块已经为你训练好了 3 个子集模型并给定了一个待解释样本 `x_demo`。请：

1. 写出 `v(S)` 的四个取值：`v_empty`、`v_A`、`v_B`、`v_AB`（空集的价值就是训练集均值 `y_tr.mean()`）；
2. 按上式算出 `phi_A` 与 `phi_B`；
3. 验证有效性公理：`v_empty + phi_A + phi_B` 是否等于 `v_AB`（误差应在 $10^{-10}$ 以内）。

In [ ]:
A, B = "MolLogP", "TPSA"

m_A  = RandomForestRegressor(n_estimators=80, random_state=0, n_jobs=-1).fit(X_tr[[A]], y_tr)
m_B  = RandomForestRegressor(n_estimators=80, random_state=0, n_jobs=-1).fit(X_tr[[B]], y_tr)
m_AB = RandomForestRegressor(n_estimators=80, random_state=0, n_jobs=-1).fit(X_tr[[A, B]], y_tr)

x_demo = X_te[[A, B]].iloc[[0]]          # 保持 DataFrame 形状，便于直接 predict
print("待解释样本：")
print(x_demo.to_string(index=False))

# 在下方填写你的代码：


---

## 第二部分：三维构象与 PubChem API（对应 `05b`）

### 🏋️ 习题 6：生成三维构象并读出 MMFF 能量

对蒽（`"c1ccc2cc3ccccc3cc2c1"`）：

1. `Chem.MolFromSmiles` 之后**先 `Chem.AddHs`**（漏掉这一步会让构象与空间位阻完全失真）；
2. 用 `params = AllChem.ETKDGv3(); params.randomSeed = 42` 调用 `AllChem.EmbedMolecule`，打印返回码；
3. 调用 `AllChem.MMFFOptimizeMolecule(mol, maxIters=1000)`，再用
   `ff = AllChem.MMFFGetMoleculeForceField(mol, AllChem.MMFFGetMoleculeProperties(mol))`
   取出 `ff.CalcEnergy()` 并打印；
4. 打印加氢前后的原子数。

- **预期输出**：嵌入返回码为 `0`；加氢前 14 个原子，加氢后 24 个；MMFF 能量在几十 kcal/mol 量级。

In [ ]:
# 在下方填写你的代码：


### 🏋️ 习题 7：用 NPR1 / NPR2 判断分子形状

对下面三个分子分别生成 10 个构象、取能量最低的那个，计算 `Descriptors3D.NPR1`、`Descriptors3D.NPR2` 和 `rdMolDescriptors.CalcPBF`：

- `Hexane`（棒状）: `"CCCCCC"`
- `Pentacene`（盘状）: `"c1ccc2cc3cc4cc5ccccc5cc4cc3cc2c1"`
- `Adamantane`（球状）: `"C1C2CC3CC1CC(C2)C3"`

整理成一张 DataFrame 打印，并回答：

1. 哪个分子的 `NPR1` 最接近 `1.0`？它对应形状三角的哪个顶点？
2. 哪个分子的 `PBF` 最接近 `0`？`PBF` 为什么能用来判断分子是不是平面的？

- **预期输出**：金刚烷的 `NPR1` 与 `NPR2` 都约等于 `1.0`（球状顶点）；并五苯的 `PBF` 约等于 `0`（完美平面）。

In [ ]:
# 在下方填写你的代码：


### 🏋️ 习题 8：用 PubChem 抓取数据并验证 Yalkowsky 方程

1. 用 PUG-REST 的 `compound/cid/<CID列表>/property/MolecularWeight,XLogP/JSON` 接口，**一次请求**拿回下面 5 个分子的分子量与 `XLogP`：
   `{"naphthalene": 931, "anthracene": 8418, "phenanthrene": 995, "pyrene": 31423, "chrysene": 9171}`；
2. 读入 `data/melting_points.csv`，取出这 5 个分子的 `Tm_C` 与 `logS`（按 `name` 列匹配）；
3. 用 PubChem 的 `XLogP`（而不是 RDKit 的 `MolLogP`）代入 Yalkowsky 方程
   $$\log S_{\text{pred}} = 0.5 - \text{XLogP} - 0.01\,(T_m - 25)$$
   打印一张对比表（列：`name`、`XLogP`、`Tm_C`、`logS_pred`、`logS_exp`、`error`），并计算 RMSE。

- **提示**：记得 `time.sleep(0.25)` 遵守 PubChem 的限速；如果网络不通，请在答案里改用 `melting_points.csv` 中的 `MolLogP` 列完成第 3 步。
- **预期输出**：这 5 个并苯分子的 RMSE 低于 1 个对数单位，且 5 个误差**全为正**（GSE 系统性地略微高估了并苯的溶解度）——对一个只有两个变量的半经验公式来说相当不错。

In [ ]:
# 在下方填写你的代码：


---

## 完成后

- 对照 `solutions/05_exercises_solutions.ipynb` 检查本套 8 道基础习题的答案；
- `05a` 与 `05b` 正文中内嵌的 6 道课堂练习（`05a-1` ~ `05a-3`、`05b-1` ~ `05b-3`）的参考解答在 `solutions/05_feature_solutions.ipynb`。